# 03 — Mega Abomasnow ex 提出練習

Simulation 部門の **初回提出練習** 用ノートブックです。

## 事前準備（Input に Add Data）

1. **cg-lib** — コンペ公式 API
2. **Mega Abomasnow ex デッキ dataset** — `deck.csv` を含む
3. **本リポジトリ（Pokemon）** — `agents/mega_abomasnow_ex.py` を読むため

## 手順

1. 下のセルを上から順に実行
2. Output から **`submission.tar.gz`** をダウンロード
3. コンペページ **My Submissions** にアップロード
4. Validation Episode を確認

詳細: [docs/submission_practice_abomasnow.md](../docs/submission_practice_abomasnow.md)

In [ ]:
import glob
from pathlib import Path

AGENT_FILE = "mega_abomasnow_ex.py"


def pick_first(matches: list[str]) -> Path | None:
    return Path(matches[0]) if matches else None


def pick_deck_csv() -> Path:
    all_decks = glob.glob("/kaggle/input/**/deck.csv", recursive=True)
    if not all_decks:
        raise FileNotFoundError("deck.csv not found. Add Mega Abomasnow ex deck dataset as Input.")
    preferred = [p for p in all_decks if "abomasnow" in p.lower()]
    if len(preferred) == 1:
        return Path(preferred[0])
    if len(preferred) > 1:
        print("WARN: multiple abomasnow deck.csv:", preferred)
        return Path(preferred[0])
    if len(all_decks) == 1:
        return Path(all_decks[0])
    print("WARN: multiple deck.csv found; using first:", all_decks)
    print("TIP: Add only one deck dataset, or use path containing 'abomasnow'.")
    return Path(all_decks[0])


def pick_agent_py() -> Path:
    patterns = [
        "/kaggle/input/**/agents/mega_abomasnow_ex.py",
        "/kaggle/input/**/Pokemon/agents/mega_abomasnow_ex.py",
        "/kaggle/input/**/mega_abomasnow_ex.py",
    ]
    for pattern in patterns:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return Path(matches[0])
    raise FileNotFoundError(
        "mega_abomasnow_ex.py not found. Add this Pokemon repo as Kaggle Input."
    )


cg_matches = glob.glob("/kaggle/input/**/cg-lib/cg", recursive=True)
cg_path = pick_first(cg_matches)
deck_path = pick_deck_csv()
agent_path = pick_agent_py()

print("cg:", cg_path)
print("deck.csv:", deck_path)
print("agent:", agent_path)

assert cg_path is not None, "cg-lib Input missing"
assert (cg_path / "api.py").is_file() or any(cg_path.glob("**/api.py")), f"cg looks invalid: {cg_path}"
assert deck_path.is_file(), deck_path
assert agent_path.is_file(), agent_path
print("OK — all inputs found")

In [ ]:
import os
import shutil
import tarfile

OUTPUT = "submission.tar.gz"

with open(agent_path, "r", encoding="utf-8") as src, open("main.py", "w", encoding="utf-8") as dst:
    dst.write(src.read())

if os.path.exists(OUTPUT):
    os.remove(OUTPUT)

with tarfile.open(OUTPUT, "w:gz") as tar:
    tar.add("main.py", arcname="main.py")
    tar.add(str(cg_path), arcname="cg")
    tar.add(str(deck_path), arcname="deck.csv")

os.remove("main.py")
print(f"Created {OUTPUT} ({os.path.getsize(OUTPUT)} bytes)")

In [ ]:
import tarfile

with tarfile.open(OUTPUT, "r:gz") as tar:
    names = tar.getnames()

print("members (first 15):", names[:15])
print("total members:", len(names))

required = {"main.py", "deck.csv"}
missing = sorted(required - set(names))
has_cg = any(n == "cg" or n.startswith("cg/") for n in names)
has_api = any("api.py" in n for n in names)

print("missing:", missing)
print("has cg:", has_cg)
print("has api.py:", has_api)

assert not missing, f"Missing in tar: {missing}"
assert has_cg, "cg/ not in tar — Validation will fail with ModuleNotFoundError: cg"
assert has_api, "cg api not found in tar"
print("OK — ready to submit", OUTPUT)

## My Submissions にアップロード

1. コンペページ → **My Submissions** タブ
2. **Upload** → 上で作った `submission.tar.gz`
3. Validation Episode を待つ

## 成功 / 失敗の見分け

| 結果 | 意味 |
|------|------|
| Validation が進み `status: ACTIVE` が続く | tar.gz 構成 OK |
| step 1 で `ModuleNotFoundError: No module named 'cg'` | `.py` 単体提出、または tar に `cg/` なし |
| `InvalidArgument` / import error | `main.py` の構文エラー |

提出回数: **1チーム1日5回まで**